In [ ]:
import pandas as pd
import numpy as np

df_tpm = pd.read_csv("../../rna-seq/star_rsem/rsem.merged.gene_tpm.tsv", sep="\t")
df_tpm.set_index("gene_id", inplace=True)
df_tpm

In [ ]:
df_geneWTA = pd.read_csv("WT_100kb_Agene_id")
df_geneWTA.columns = ["gene_id"]
df_geneWTA.set_index("gene_id", inplace=True)
df_tpmWTA = df_tpm.loc[df_geneWTA.index].drop(["transcript_id(s)", "TGEV_REP1", "TGEV_REP2", "TGEV_REP3"], axis=1)
df_tpmWT_A = df_tpmWTA[["WT_REP1","WT_REP2", "WT_REP1"]].mean(axis=1)

In [ ]:
df_geneWTB = pd.read_csv("WT_100kb_Bgene_id")
df_geneWTB.columns = ["gene_id"]
df_geneWTB.set_index("gene_id", inplace=True)
df_tpmWTB = df_tpm.loc[df_geneWTB.index].drop(["transcript_id(s)", "TGEV_REP1", "TGEV_REP2", "TGEV_REP3"], axis=1)
df_tpmWT_B = df_tpmWTB[["WT_REP1","WT_REP2", "WT_REP1"]].mean(axis=1)
df_tpmWT_B

In [ ]:
df = pd.concat([df_tpmWT_A, df_tpmWT_B], axis=1)
df.columns = ["WT_A", "WT_B"]
df_p = df.melt()
df_p.columns = ["Type", "Tpm"]
df_p['Tpm'] = df_p['Tpm'].apply(lambda x: np.log(x + 0.00001))
df_p

In [ ]:
from scipy.stats import ranksums
z, p = ranksums(df['WT_A'].dropna(), df['WT_B'].dropna())
print(z, p)

In [ ]:
import plotly.express as px
import plotly.io as pio
import numpy as np

fig = px.box(df_p.dropna(), 
             x="Type",
             y="Tpm", 
             color="Type"
            )

fig.update_layout(
    width=500,
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# Add title and set title position
fig.update_layout(
    title={
        'text': "A Vs B",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)


fig.update_yaxes(range=[-15, 15])

# Save figure as PDF file
pio.write_image(fig, 'WTboxplot.pdf')
fig.show()